Cargar dataset

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [ ]:
SEED = 123
DATA_PATH = Path("../data/raw/enfermedad_cardiovascular.csv")
df = pd.read_csv(DATA_PATH)
filas = df.shape[0]
print(df.shape)

print("Valores nulos: ", df.isnull().sum().sum())
df_resumen_nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().mean() * 100).round(2)
})
df_resumen_nulos = df_resumen_nulos[df_resumen_nulos['Nulos'] > 0].sort_values(by='Nulos', ascending=False)

df_resumen_nulos

(2254, 28)
Valores nulos:  584


,Nulos,Porcentaje (%)
horas_sueno_diarias,180,7.99
proteina_c_reactiva,157,6.97
colesterol_ldl,135,5.99
glucosa_ayunas,112,4.97


# Procesos comunes

Quitar filas o ponerlas NaN

In [ ]:
# Eliminar menores de edad
df = df[df['edad'] >= 18]
print(filas - df.shape[0], "filas eliminadas por ser menores de edad. Hay ", df.shape[0], "filas restantes.")
filas = df.shape[0]
# Elimininar pacientes con más años fumando que edad
df = df[df['edad'] >= df['anos_fumador']]
print(filas - df.shape[0], "filas eliminadas por tener más años fumando que edad. Hay ", df.shape[0], "filas restantes.")
filas = df.shape[0]
# Poner a NaN los valores negativos o 0 de colesterol_hdl
cond = (df['colesterol_hdl'] <= 0).sum()
df.loc[df['colesterol_hdl'] <= 0, 'colesterol_hdl'] = np.nan
print( cond, "valores con colesterol_hdl negativo o 0 se han puesto a NaN.")
# Poner a NaN los valores negativos o 0 de trigliceridos
cond = (df['trigliceridos'] <= 0).sum()
df.loc[df['trigliceridos'] <= 0, 'trigliceridos'] = np.nan
print( cond, "valores con trigliceridos negativos o 0 se han puesto a NaN.")
# Poner a NaN los valores negativos de consumo_alcohol_semanal
cond = (df['consumo_alcohol_semanal'] < 0).sum()
df.loc[df['consumo_alcohol_semanal'] < 0, 'consumo_alcohol_semanal'] = np.nan
print( cond, "valores con consumo de alcohol semanal negativo se han puesto a NaN.")
# Poner a NaN si IMC < 13
cond = (df['indice_masa_corporal'] < 13).sum()
df.loc[df['indice_masa_corporal'] < 13, 'indice_masa_corporal'] = np.nan
print( cond, "valores con IMC menor que 13 se han puesto a NaN.")
# Poner a NaN si tiempo semanal es mayor que 7
cond = (df['frecuencia_ejercicio_semanal'] > 7).sum()
df.loc[df['frecuencia_ejercicio_semanal'] > 7, 'frecuencia_ejercicio_semanal'] = np.nan
print( cond, "valores con tiempo semanal mayor que 7 se han puesto a NaN.")
# Poner a NaN si hemoglobina glicosilada es menor del 3%
cond = (df['hemoglobina_glicosilada'] < 3).sum()
df.loc[df['hemoglobina_glicosilada'] < 3, 'hemoglobina_glicosilada'] = np.nan
print( cond, "valores con hemoglobina glicosilada menor que 3% se han puesto a NaN.")
# Poner a NaN si la presión sistólica es menor que 60
cond = (df['presion_sistolica'] < 60).sum()
df.loc[df['presion_sistolica'] < 60, 'presion_sistolica'] = np.nan
print( cond, "valores con presión sistólica menor que 60 se han puesto a NaN.")
# Poner a NaN si la presión diastólica es menor que 40
cond = (df['presion_diastolica'] < 40).sum()
df.loc[df['presion_diastolica'] < 40, 'presion_diastolica'] = np.nan
print( cond, "valores con presión diastólica menor que 40 se han puesto a NaN.")
# Quitar la columna de colesterol_total
df = df.drop('colesterol_total', axis=1)

# Poner a NaN si presión diastólica >= sistólica
cond = (df['presion_diastolica'] >= df['presion_sistolica']).sum()
df.loc[df['presion_diastolica'] >= df['presion_sistolica'], ['presion_sistolica', 'presion_diastolica']] = np.nan
print(cond, "pacientes con presión diastólica >= sistólica se han puesto a NaN en ambas presiones.")
# Poner a NaN si IMC es mayor que 35 y el perímetro de la cintura es menor que 70
cond = ((df['indice_masa_corporal'] > 35) & (df['perimetro_cintura'] < 70)).sum()
df.loc[(df['indice_masa_corporal'] > 35) & (df['perimetro_cintura'] < 70), ['indice_masa_corporal', 'perimetro_cintura']] = np.nan
print( cond, "valores con IMC mayor que 35 y cintura menor que 70 se han puesto a NaN en ambas columnas.")
# Poner a NaN si IMC es menor que 18.5 y el perímetro de la cintura es mayor que 110
cond = ((df['indice_masa_corporal'] < 18.5) & (df['perimetro_cintura'] > 110)).sum()
df.loc[(df['indice_masa_corporal'] < 18.5) & (df['perimetro_cintura'] > 110), ['indice_masa_corporal', 'perimetro_cintura']] = np.nan
print( cond, "valores con IMC menor que 18.5 y cintura mayor que 110 se han puesto a NaN en ambas columnas.")
# Poner NaN si la hemoglobina glicosilada es menor que 5.5% y la glucosa en ayunas es mayor que 180
cond = ((df['hemoglobina_glicosilada'] < 5.5) & (df['glucosa_ayunas'] > 180)).sum()
df.loc[(df['hemoglobina_glicosilada'] < 5.5) & (df['glucosa_ayunas'] > 180), ['hemoglobina_glicosilada', 'glucosa_ayunas']] = np.nan
print( cond, "valores con hemoglobina glicosilada menor que 5.5% y glucosa en ayunas mayor que 180 se han puesto a NaN en ambas columnas.")
# Poner a NaN si la hemoglobina glicosilada es mayor que 9% y la glucosa en ayunas es menor que 80
cond = ((df['hemoglobina_glicosilada'] > 9) & (df['glucosa_ayunas'] < 80)).sum()
df.loc[(df['hemoglobina_glicosilada'] > 9) & (df['glucosa_ayunas'] < 80), ['hemoglobina_glicosilada', 'glucosa_ayunas']] = np.nan
print( cond, "valores con hemoglobina glicosilada mayor que 9% y glucosa en ayunas menor que 80 se han puesto a NaN en ambas columnas.")

4 filas eliminadas por ser menores de edad. Hay  2250 filas restantes.
0 filas eliminadas por tener más años fumando que edad. Hay  2250 filas restantes.
4 valores con colesterol_hdl negativo o 0 se han puesto a NaN.
4 valores con trigliceridos negativos o 0 se han puesto a NaN.
177 valores con consumo de alcohol semanal negativo se han puesto a NaN.
19 valores con IMC menor que 13 se han puesto a NaN.
41 valores con tiempo semanal mayor que 7 se han puesto a NaN.
1 valores con hemoglobina glicosilada menor que 3% se han puesto a NaN.
44 pacientes con presión diastólica >= sistólica se han puesto a NaN en ambas presiones.
18 valores con IMC mayor que 35 y cintura menor que 70 se han puesto a NaN en ambas columnas.
16 valores con IMC menor que 18.5 y cintura mayor que 110 se han puesto a NaN en ambas columnas.
1 valores con hemoglobina glicosilada menor que 5.5% y glucosa en ayunas mayor que 180 se han puesto a NaN en ambas columnas.
0 valores con hemoglobina glicosilada mayor que 9% y 

Dummies

In [ ]:
encoder_nominal = OneHotEncoder(drop='first', handle_unknown='ignore') # Para tipo_alimentacion y antecedentes_familiares_tipo
encoder_ordinal = OrdinalEncoder(categories=[['Sedentario', 'Ligeramente activo', 'Moderadamente activo', 'Muy activo']]) # Para nivel_actividad

(2250, 28)


Para ver los nulos tras la limpieza

In [ ]:
print("Valores nulos: ", df.isnull().sum().sum())
df_resumen_nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().mean() * 100).round(2)
})
df_resumen_nulos = df_resumen_nulos[df_resumen_nulos['Nulos'] > 0].sort_values(by='Nulos', ascending=False)

df_resumen_nulos

Valores nulos:  987


,Nulos,Porcentaje (%)
horas_sueno_diarias,180,8.00
consumo_alcohol_semanal,177,7.87
proteina_c_reactiva,157,6.98
colesterol_ldl,134,5.96
glucosa_ayunas,113,5.02
indice_masa_corporal,53,2.36
presion_sistolica,44,1.96
presion_diastolica,44,1.96
frecuencia_ejercicio_semanal,41,1.82
perimetro_cintura,34,1.51
